In [1]:
import pandas as pd, numpy as np, requests, json, os, warnings
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay)
import joblib
from mplsoccer import Pitch
warnings.filterwarnings('ignore')
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('figures', exist_ok=True)
os.makedirs('dashboard', exist_ok=True)

In [2]:
BASE = "https://premier.72-60-245-2.sslip.io"

def load_or_download(path, url):
    if os.path.exists(path):
        print(f"[caché] Cargando {path}")
        return pd.read_csv(path)
    print(f"[descarga] {url}")
    df = pd.read_csv(url)
    df.to_csv(path, index=False)
    print(f"[guardado] {path} — {df.shape}")
    return df

players = load_or_download('data/players.csv', f"{BASE}/export/players")
matches = load_or_download('data/matches.csv', f"{BASE}/export/matches")
events  = load_or_download('data/events.csv',  f"{BASE}/export/events")

[caché] Cargando data/players.csv
[caché] Cargando data/matches.csv
[caché] Cargando data/events.csv


In [3]:
for name, df in [('players', players), ('matches', matches), ('events', events)]:
    print(f"\n=== {name} ===")
    print(f"Shape: {df.shape}")
    print(f"Nulos: {df.isnull().sum()[df.isnull().sum()>0].to_dict()}")
    display(df.head(2))


=== players ===
Shape: (822, 37)
Nulos: {'chance_of_playing_next_round': 218, 'news': 515}


,id,first_name,second_name,web_name,team,team_short,position,price,status,total_points,...,ict_index,form,points_per_game,selected_by_percent,bonus,bps,transfers_in,transfers_out,chance_of_playing_next_round,news
0,430,Erling,Haaland,Haaland,Man City,MCI,FWD,14.6,a,197,...,245.9,2.5,6.8,58.6,35,786,6901750,3106293,100.0,NaN
1,82,Antoine,Semenyo,Semenyo,Man City,MCI,MID,8.3,a,174,...,214.5,5.2,6.0,56.2,17,569,11948832,6186420,100.0,NaN



=== matches ===
Shape: (291, 41)
Nulos: {}


,id,date,time,home_team,away_team,fthg,ftag,ftr,hthg,htag,...,maxd,maxa,avgh,avgd,avga,total_goals,goal_diff,implied_prob_h,implied_prob_d,implied_prob_a
0,187,01/01/2026,17:30,Crystal Palace,Fulham,1,1,D,1,0,...,3.4,3.5,2.17,3.34,3.40,2,0,0.465,0.303,0.294
1,188,01/01/2026,17:30,Liverpool,Leeds,0,0,D,0,0,...,4.6,6.4,1.53,4.41,5.83,0,0,0.645,0.222,0.182



=== events ===
Shape: (444252, 19)
Nulos: {'second': 933, 'player_name': 4629, 'player_id': 4629, 'end_x': 148867, 'end_y': 148867, 'goal_mouth_y': 437054, 'goal_mouth_z': 437054}


,id,match_id,minute,second,period,event_type,outcome,team_name,player_name,player_id,x,y,end_x,end_y,goal_mouth_y,goal_mouth_z,is_touch,is_shot,is_goal
0,6433,1,0,0.0,FirstHalf,Start,Successful,Liverpool,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0,0,0
1,6434,1,0,0.0,FirstHalf,Start,Successful,Bournemouth,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0,0,0


In [4]:
if 'as_' in matches.columns:
    matches.rename(columns={'as_': 'away_shots'}, inplace=True)
    print("✅ as_ renombrada a away_shots")

# Rename columns to standard ones used in the code
cols_to_rename = {'ftr': 'result', 'fthg': 'home_goals', 'ftag': 'away_goals', 'hy': 'home_yellow_cards'}
matches.rename(columns={k: v for k, v in cols_to_rename.items() if k in matches.columns}, inplace=True)

matches['date'] = pd.to_datetime(matches['date'], format='%d/%m/%Y')
print("✅ Fechas convertidas")

team_map = {
    'Man United': 'Man Utd',
    "Nott'm Forest": 'Nottingham Forest',
    'Spurs': 'Tottenham',
    'Wolves': 'Wolverhampton',
}
for col in ['home_team', 'away_team']:
    if col in matches.columns:
        matches[col] = matches[col].replace(team_map)
print("✅ team_map aplicado")

✅ as_ renombrada a away_shots
✅ Fechas convertidas
✅ team_map aplicado


In [5]:
events_spatial = events[(events['x'] != 0) | (events['y'] != 0)].copy()
print(f"events original: {len(events)} | events_spatial: {len(events_spatial)}")

events original: 444252 | events_spatial: 433808


In [6]:
shots = events_spatial[events_spatial['is_shot'] == True].copy()
print(f"Tiros totales: {len(shots)} | Goles: {shots['is_goal'].sum()}")
print(f"Tasa de conversión: {shots['is_goal'].mean()*100:.1f}%")

Tiros totales: 7198 | Goles: 807
Tasa de conversión: 11.2%


In [7]:
if 'qualifiers' in shots.columns:
    q = shots['qualifiers'].astype(str)
else:
    q = pd.Series('', index=shots.index)

shots['is_big_chance']  = q.str.contains('BigChance',  na=False).astype(int)
shots['is_header']      = q.str.contains('"Head"',     na=False).astype(int)
shots['is_right_foot']  = q.str.contains('RightFoot',  na=False).astype(int)
shots['is_left_foot']   = q.str.contains('LeftFoot',   na=False).astype(int)
shots['is_penalty']     = q.str.contains('"Penalty"',  na=False).astype(int)
shots['is_counter']     = q.str.contains('FastBreak',  na=False).astype(int)
shots['from_corner']    = q.str.contains('FromCorner', na=False).astype(int)
shots['is_volley']      = q.str.contains('Volley',     na=False).astype(int)
shots['first_touch']    = q.str.contains('FirstTouch', na=False).astype(int)

qualifier_cols = ['is_big_chance','is_header','is_right_foot','is_left_foot',
                  'is_penalty','is_counter','from_corner','is_volley','first_touch']
print(shots[qualifier_cols].sum().sort_values(ascending=False))

is_big_chance    0
is_header        0
is_right_foot    0
is_left_foot     0
is_penalty       0
is_counter       0
from_corner      0
is_volley        0
first_touch      0
dtype: int64


In [8]:
shots['distance_to_goal'] = np.sqrt((100 - shots['x'])**2 + (50 - shots['y'])**2)
shots['angle_to_goal']    = np.arctan2(np.abs(50 - shots['y']), 100 - shots['x'])
shots['x_norm']           = shots['x'] / 100
shots['y_centered']       = np.abs(shots['y'] - 50) / 50
print(shots[['x','y','distance_to_goal','angle_to_goal','is_goal']].describe())

                 x            y  distance_to_goal  angle_to_goal      is_goal
count  7198.000000  7198.000000       7198.000000    7198.000000  7198.000000
mean     85.761198    50.303834         18.188906       0.575127     0.112114
std       9.089237    11.637737          9.488567       0.370596     0.315529
min       0.500000     0.500000          0.600000       0.000000     0.000000
25%      80.800000    42.700000         11.431425       0.249826     0.000000
50%      87.200000    50.100000         17.837741       0.560471     0.000000
75%      91.500000    58.400000         23.809609       0.855486     0.000000
max      99.600000    99.200000         99.551444       1.562650     1.000000


In [9]:
shots.to_csv('data/shots_processed.csv', index=False)
matches.to_csv('data/matches_clean.csv', index=False)
print("✅ shots_processed.csv y matches_clean.csv guardados")

✅ shots_processed.csv y matches_clean.csv guardados
